# An Introduction to Hybrid Quantum Neural Networks — Quantum Machine Learning and Data Analysis
$\renewcommand{\ket}[1]{|#1\rangle}\renewcommand{\bra}[1]{\langle#1|}$

---

Quantum machine learning (QML) applies learning methods to quantum data, quantum models, or hybrid quantum–classical workflows. This notebook focuses on **neural networks (NNs)**, **quantum neural networks (QNNs)**, and **hybrid quantum neural networks (HQNNs)**. It combines foundational theory with hands-on CUDA-Q and PyTorch exercises for classification and time-series forecasting.

HQNNs can support supervised, unsupervised, reinforcement, and generative learning. This lesson begins with supervised learning, where labeled examples are used to learn predictions from inputs.

This lesson was co-developed with researchers at Chung Yuan Christian University (CYCU) and is motivated by their paper [*Solar Irradiance Forecasting Using a Hybrid Quantum Neural Network*](https://doi.org/10.1109/ACCESS.2024.3472053). The final workflow is a simplified, single-qubit forecasting example inspired by that work.

**What You Will Do:**
* Review the building blocks of classical neural networks
* Explain how quantum and classical layers combine in an HQNN
* Build a classical neural network with PyTorch to classify points inside or outside a circle
* Build a QNN with CUDA-Q for the same circle-classification problem
* Integrate a single-qubit quantum layer into a PyTorch LSTM workflow for sea-surface-temperature forecasting

**Prerequisites:**
* Python and Jupyter notebook familiarity
* Basic knowledge of quantum states, gates, and superposition
* Familiarity with variational quantum algorithms

**Key Terminology:**
* Neural network (NN)
* Quantum neural network (QNN)
* Hybrid quantum neural network (HQNN)
* Long short-term memory (LSTM)
* Ansatz
* Parameter-shift rule
* Barren plateau

**CUDA-Q Syntax:**
* [`@cudaq.kernel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.kernel) — defines quantum circuits as Python functions
* [`cudaq.qvector`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.qvector) — allocates qubits inside a kernel
* [`cudaq.observe`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.observe) — computes expectation values of spin operators and supports batched kernel arguments
* [`cudaq.spin`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#spin-operators) — constructs Pauli spin operators and Hamiltonians
* [`cudaq.set_target`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.set_target) — selects a simulation or hardware backend
* Parameterized gates: `ry`, `rx`, `rz`, and `x.ctrl` (CNOT)



<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 12px 15px 12px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900;">&#9889; GPU Required:</span>** This notebook requires an NVIDIA GPU for the CUDA-Q `nvidia` target and PyTorch training.

</div>

In [ ]:
## Instructions for Google Colab. You can ignore this cell if you have CUDA-Q
## set up locally with all required files on your system.
## Uncomment the lines below and execute this cell to install CUDA-Q.

!pip install cudaq -q
!pip install -q numpy torch scipy tqdm matplotlib pandas ipython

> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).


In [ ]:
# Scientific computing
import numpy as np

# Machine learning
import torch
from torch.autograd import Function
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

# Optimization and progress reporting
from scipy.optimize import minimize
from tqdm.auto import tqdm

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# CUDA-Q
import cudaq
from cudaq import spin

SEED = 111

torch.manual_seed(SEED)
cudaq.set_random_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
cudaq.set_target("nvidia" if torch.cuda.is_available() else "qpp-cpu")
print(f"PyTorch device: {device}")
print(f"CUDA-Q target: {cudaq.get_target().name}")

---
## 1. A Brief Introduction to Classical Neural Networks

Classical neural networks are parameterized models that can approximate broad classes of nonlinear functions when given a suitable architecture, objective, and training data.

The key word here is learning.  Often, a function is a black box that turns a set of inputs into outputs. Consider use cases like:

* A pharmaceutical scientist trying to determine how drug properties (inputs) can predict candidate drugs (outputs).
* An investor trying to determine how a host of political and economic factors (inputs) inform which stocks are best to buy (outputs).
* A retail grocery store wants to figure out given a shopper's characteristics (inputs) what coupons to send them (outputs) to maximize profit.

In each case, the target function is so complex and abstract that it is useful to learn the function from data instead of defining a model *a priori*. The construction of a neural network makes it well suited to learning such functions. This lesson introduces only the concepts needed for HQNNs; for broader AI coursework, see the [NVIDIA Deep Learning Institute training catalog](https://www.nvidia.com/en-us/training/).

Many feed-forward neural networks organize neurons into layers. In the fully connected example used here, each neuron receives values from every neuron in the preceding layer. Other architectures use different connectivity patterns. See the figure below.


Each connection has a weight $w_j$ and each intermediate node (often said to be in a hidden layer as a user only "sees" the inputs and the outputs) has an associated bias term $b_i$. These are the parameters that are actually learned, but more on that later.  The values of nodes the next layer are computed by taking the values of nodes it connects to in the previous layer, multiplying these by their respective weights, adding the bias term, and then applying an activation function to that result. 

<img src="https://github.com/osbama/Phys710/blob/master/hands-on/hands-on-7-images/nn_math.png?raw=1" alt="A feed-forward neural network with input, hidden, and output layers connected by weighted edges" width="65%"/> 

Nonlinear activation functions between linear layers allow the network to represent relationships that a sequence of linear operations alone cannot capture. Neural-network design also involves choices such as layer width, depth, connectivity, and activation functions. More sophisticated architectures are beyond the scope of this lesson.



**Exercise 1:**

In practice, NNs become so large that they are never coded by hand.  Instead, tools like PyTorch make it easy to set up a NN and train it with ease.  Your task is to use the PyTorch code in the following cells to train a NN that can classify if a point on the x,y plane (between -1 and 1 in both dimensions) is within a circle of radius 0.5.


<img src="https://github.com/osbama/Phys710/blob/master/hands-on/hands-on-7-images/circle_problem.png?raw=1" alt="Points in a square labeled by whether they fall inside or outside a centered circle" width="30%"/>

**a.** First, you need to generate random data to train the model (complete the sections marked TODO). We have not discussed this yet, but the structure of the training data is critical for training a useful model. First, randomly generate pairs of x,y coordinates and a list of labels for points if they do (0) or don't (1) fall within a circle of radius 0.5. Train the NN and look at the plotted results for the test set. Does anything look odd?


**b.** If you did not see this, rerun the cell a few times. You may notice that the model learns to always predict that a point is outside the circle because a circle of radius 0.5 occupies just under 20% of the square's area. The model can therefore learn nothing about the circle's shape and still obtain approximately 80% accuracy. Improve the data generation so that half the points are inside the circle and half are outside. The model is now trained on a balanced dataset, so an “always outside” prediction no longer achieves high accuracy. Does the model learn the circle?


**c.** Find the part of the code that defines the NN structure. If this is not clear, use AI to help. Increase the hidden-layer width from 8 nodes to 16 and then 32. Compare the resulting accuracy across several runs. How many trainable parameters does each model contain? You can use AI to help print the parameter count or calculate it by hand. Wider models have more capacity, but they are not guaranteed to achieve higher test accuracy in every run.




In [ ]:
# Circle we are classifying: radius 0.5 (radius_sq = 0.25).
CIRCLE_RADIUS = 0.5
RADIUS_SQ = CIRCLE_RADIUS ** 2  # 0.25

def generate_data(num_samples=1000):
    x_data = []
    y_labels = []
    half = num_samples // 2

    # ##TODO## First sample points uniformly from the square and label them.
    # After observing the class imbalance, revise this function so half of
    # the returned points are inside the circle and half are outside.
    raise NotImplementedError("Complete Exercise 1 data generation")
            
    x_data = np.array(x_data)
    y_labels = np.array(y_labels)
    
    # Shuffle to mix classes for training
    indices = np.random.permutation(num_samples)
    return x_data[indices], y_labels[indices]



num_samples = 1000  # change this to vary total data size; test set is always 20% of full data
all_inputs, all_targets = generate_data(num_samples)
test_size = int(num_samples * 0.20)
train_size = num_samples - test_size
training_inputs = all_inputs[:train_size]
training_targets = all_targets[:train_size]
test_inputs = all_inputs[train_size:]
test_targets = all_targets[train_size:]

In [ ]:
# Convert data to Tensors (using data from previous cell)
X_train = torch.FloatTensor(training_inputs)
Y_train = torch.FloatTensor(training_targets).view(-1, 1)
X_test = torch.FloatTensor(test_inputs)
Y_test = torch.FloatTensor(test_targets).view(-1, 1)

model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

# Define optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
criterion = nn.BCELoss()  
mse_criterion = nn.MSELoss() 

# Training Loop
loss_history = []
print(f"Training Classical NN on {len(X_train)} samples...")

for epoch in range(400):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, Y_train)
    
    # Track MSE
    mse_val = mse_criterion(outputs, Y_train).item()
    loss_history.append(mse_val)
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/400] - MSE: {mse_val:.4f}", end='\r')


# ------------------------------Visualizing Convergence and Classification--------------------------------

plt.style.use('dark_background')
fig, (ax_loss, ax_plot) = plt.subplots(1, 2, figsize=(18, 8))

# --- Left Plot: Loss Convergence ---
ax_loss.plot(loss_history, color='#76B900', linewidth=2, label='Classical MSE')
ax_loss.set_title('Classical Loss Convergence', fontsize=16)
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('Mean Squared Error')
ax_loss.grid(alpha=0.2)
ax_loss.legend()

# --- Right Plot: Classification ---
with torch.no_grad():
    raw_preds = model(X_test)
    # Binary prediction: 1 if >= 0.5, else 0
    preds = (raw_preds >= 0.5).float().numpy().flatten()

test_inputs_np = X_test.numpy()
test_targets_np = Y_test.numpy().flatten()

# Predicted Outside (1)
idx_1 = (preds == 1)
ax_plot.scatter(test_inputs_np[idx_1, 0], test_inputs_np[idx_1, 1], 
                color='#76B900', s=50, label='Pred: Outside (Green)', zorder=3)

# Predicted Inside (0) 
idx_0 = (preds == 0)
ax_plot.scatter(test_inputs_np[idx_0, 0], test_inputs_np[idx_0, 1], 
                color='#A020F0', s=50, label='Pred: Inside (Purple)', zorder=3)

# Mark Errors with Red Rings
errors = (preds != test_targets_np)
ax_plot.scatter(test_inputs_np[errors, 0], test_inputs_np[errors, 1], 
                facecolors='none', edgecolors='#FF0000', s=150, linewidth=2, 
                label='Errors', zorder=4)

# Circle boundary (must match labels: CIRCLE_RADIUS from data cell)
boundary = patches.Circle((0, 0), CIRCLE_RADIUS, fill=False, edgecolor='white', 
                          linestyle='--', linewidth=2, alpha=0.5)
ax_plot.add_patch(boundary)

accuracy = (np.sum(preds == test_targets_np) / len(test_targets_np)) * 100
ax_plot.set_title(f'Classical Classification (Acc: {accuracy:.2f}%)', fontsize=16)
ax_plot.set_xlim(-1.1, 1.1)
ax_plot.set_ylim(-1.1, 1.1)
ax_plot.set_aspect('equal')
ax_plot.legend(loc='upper right', bbox_to_anchor=(1.4, 1))

plt.tight_layout()
plt.show()

One advantage of classical neural networks is that their parameter gradients can be computed efficiently using **backpropagation**. In PyTorch, calling `loss.backward()` triggers automatic differentiation through the recorded classical tensor operations. The optimizer then uses those gradients when `optimizer.step()` updates the model parameters.


Overall, it is quite incredible that such a conceptually simple construction is the foundation for the incredible results we can produce with AI models today.


---
## 2. From Classical to Quantum Neural Networks

An $N$-qubit pure state is described by $2^N$ complex amplitudes. This high-dimensional representation can encode correlations that may be useful for learning certain functions. However, the size of the Hilbert space alone does not establish that a quantum model is efficient or provides an advantage over classical methods. Determining when a quantum model offers a practical benefit remains an active area of research.

A quantum NN generally looks like the figure below.  


<img src="https://github.com/osbama/Phys710/blob/master/hands-on/hands-on-7-images/qnn.png?raw=1" alt="Quantum neural network workflow with data encoding, parameterized quantum layers, measurement, and classical optimization" width="60%"/> 


First, classical data must be encoded so that it can be processed by a quantum computer. This can be one of the hardest parts of a QML workflow. For some encodings, preparing the quantum state requires work proportional to the amount of classical data, making data loading intractably expensive and potentially eliminating any downstream quantum advantage. Quantum random access memory, or qRAM, is a proposed mechanism for coherently accessing stored data and could accelerate certain data-loading and state-preparation tasks, but it is not a general name for quantum data encoding, and scalable qRAM is not currently available. Practical approaches such as angle encoding avoid assuming qRAM, while amplitude encoding and other schemes have their own state-preparation and resource tradeoffs.

After encoding, the QNN applies parameterized single-qubit rotations and entangling gates. The rotation angles serve as trainable circuit parameters. Entangling gates allow information encoded on different qubits to influence a shared readout. Once the qubits are correlated, subsequent rotations can change multiple computational-basis amplitudes. This may increase the circuit's expressivity, but it does not guarantee better accuracy, easier training, or fewer parameters than a classical model.

After some set number of parameterized and entangling layers, a qubit observable is chosen as the model's readout. In Exercise 2, we read qubit 1 in the computational basis by computing the expectation value of $Z_1$. The $Z$ measurement has value $+1$ when the qubit is measured as 0 and $-1$ when it is measured as 1. Therefore, $\langle Z_1\rangle=p(0)-p(1)$ and $p(0)+p(1)=1$, giving $p(1)=(1-\langle Z_1\rangle)/2$ and $p(0)=(1+\langle Z_1\rangle)/2$. On quantum hardware, or whenever a finite shot count is requested and used by the backend, the expectation value is estimated from the observed proportions of 0 and 1. In this exercise, the state-vector simulator instead computes the expectation value exactly. CUDA-Q evaluates this observable for all encoded data points in one batched `cudaq.observe` call, avoiding hundreds of separate Python calls while producing deterministic readout probabilities.

The "correct" circuit parameters are found much like the parameters of a classical NN: a loss is calculated from these output probabilities and the parameters are updated classically.  




**Exercise 2:<**

Now solve the circle-classification problem with a QNN.


**a.** Build a QNN CUDA-Q kernel where the x and y data points are encoded as $R_Y$ rotations on qubits 0 and 1, respectively. Multiply each data value by $\pi$, so the encoding angles lie within $[-\pi,\pi]$.


**b.** Add three parameterized layers. Each layer applies parameterized $R_Y$, $R_X$, and $R_Z$ gates to each qubit, followed by a CNOT gate. Use the $Z_1$ expectation value to compute the probability of measuring qubit 1 as 1, and complete the sections marked TODO or FIXME in the remaining workflow. Train the model. Are you able to improve on the 50% baseline accuracy?


**c.** To explore a smaller, unentangled circuit, remove the final two parameterized layers and comment out the CNOT gate in the remaining layer. The reduced circuit uses only six trainable parameters, so also change `num_parameters` from 18 to 6 before retraining. Because this changes both circuit depth and entanglement, the experiment does not isolate either factor by itself. The model still reads only qubit 1, so removing the CNOT prevents the feature encoded exclusively on qubit 0 from influencing the prediction. What structure do you observe in the resulting classifications?

**Note:** After modifying `num_parameters`, re-run this entire cell from the top so that `initial_params` is resized correctly.




Before diving into the mechanics, it is worth pausing on a question a machine learning student naturally asks: if a classical neural network just learned to classify points inside a circle, why introduce quantum layers at all? The honest answer is that we do not yet know which problems will benefit from quantum machine learning in practice. A few plausible motivations guide current research: quantum circuits may implement inductive biases — implicit assumptions about structure — that are expensive to replicate classically; some tasks involve quantum-origin data (measurements from quantum sensors or quantum chemistry simulations) that may be processed more naturally on a quantum device; and certain quantum models may require fewer parameters to express specific relationships. None of these translate into demonstrated advantage today, but learning the mechanics now puts you in a position to evaluate claims and contribute as the field matures.

In [ ]:
cudaq.set_target('nvidia')

num_parameters = 18 # Change this to set number of variational parameters in kernel

@cudaq.kernel
def circle_classifier(theta: list[float], encoded_data: list[float]):
    
    q = cudaq.qvector(2)
    # ##TODO## Encode both features with RY gates, then add three
    # parameterized RY/RX/RZ layers separated by CNOT gates.


# --------------Cost function--------------------------------------
cost_history = []
def cost_function(theta):
    # Each row supplies one encoded data point and one copy of the current
    # variational parameters to CUDA-Q's broadcast execution.
    encoded_batch = np.asarray(training_inputs, dtype=float) * np.pi
    theta_batch = np.repeat(
        np.asarray(theta, dtype=float)[None, :],
        len(encoded_batch),
        axis=0,
    )
    
    # ##TODO## Use one batched cudaq.observe call to evaluate Z on qubit 1,
    # convert each expectation value to p(class 1), and compute the MSE.
    avg_error = None  # TODO: compute average error across the batch
    raise NotImplementedError("Complete the Exercise 2 cost function")

    cost_history.append(avg_error)
    print(f"Iteration {len(cost_history)} - MSE: {avg_error:.4f}", end='\r')
    return avg_error

# -------Training Loop-----------------------------
initial_params = np.random.uniform(-np.pi, np.pi, num_parameters) 
print(f"Training on {len(training_inputs)} samples...")
res = minimize(cost_function, initial_params, method='COBYLA', options={'maxiter': 60})
final_params = list(res.x)
print("\nTraining Complete.")



# --------------Visualizing Convergence and Classification----------------
plt.style.use('dark_background')
fig, (ax_loss, ax_plot) = plt.subplots(1, 2, figsize=(18, 8))

# --- Left Plot: Loss Convergence ---
ax_loss.plot(cost_history, color='#76B900', linewidth=2, label='MSE Loss')
ax_loss.set_title('Loss Function Convergence', fontsize=16)
ax_loss.set_xlabel('Iteration')
ax_loss.set_ylabel('Mean Squared Error')
ax_loss.grid(alpha=0.2)
ax_loss.legend()

# --- Right Plot: Classification ---
encoded_test_batch = np.asarray(test_inputs, dtype=float) * np.pi
final_params_batch = np.repeat(
    np.asarray(final_params, dtype=float)[None, :],
    len(encoded_test_batch),
    axis=0,
)
# ##TODO## Evaluate the trained circuit on the test batch and convert the
# expectation values into integer class predictions named `preds`.
preds = None  # TODO: compute predictions for the test set
preds = None  # TODO: compute predictions for the test set
raise NotImplementedError("Complete Exercise 2 test-set prediction")
test_targets = np.array(test_targets)

# Predicted Outside (1) 
idx_1 = (preds == 1)
ax_plot.scatter(test_inputs[idx_1, 0], test_inputs[idx_1, 1], 
                color='#76B900', s=50, label='Pred: Outside (Green)', zorder=3)

# Predicted Inside (0) 
idx_0 = (preds == 0)
ax_plot.scatter(test_inputs[idx_0, 0], test_inputs[idx_0, 1], 
                color='#A020F0', s=50, label='Pred: Inside (Purple)', zorder=3)

# Mark Errors with Red Rings
errors = (preds != test_targets)
ax_plot.scatter(test_inputs[errors, 0], test_inputs[errors, 1], 
                facecolors='none', edgecolors='#FF0000', s=150, linewidth=2, 
                label='Errors', zorder=4)

# Circle Boundary
boundary = patches.Circle((0, 0), 0.5, fill=False, edgecolor='white', 
                          linestyle='--', linewidth=2, alpha=0.5)
ax_plot.add_patch(boundary)

accuracy = (np.sum(preds == test_targets) / len(test_targets)) * 100
ax_plot.set_title(f'Final Classification (Acc: {accuracy:.2f}%)', fontsize=16)
ax_plot.set_xlim(-1.1, 1.1)
ax_plot.set_ylim(-1.1, 1.1)
ax_plot.set_aspect('equal')
ax_plot.legend(loc='upper right', bbox_to_anchor=(1.4, 1))

plt.tight_layout()
plt.show()

This example demonstrates that a small QNN can produce useful predictions with relatively few trainable parameters. Scaling the approach introduces important challenges. This circuit uses one qubit per feature, which may consume limited qubit resources inefficiently, and additional features may require more gates or parameters.

A QNN's trainability depends on its **ansatz**—the parameterized circuit architecture—as well as its depth, parameter initialization, measured observable, cost function, and noise. Some sufficiently deep or highly expressive circuits can exhibit **barren plateaus**, which are large regions of the optimization landscape where gradients become extremely small. Additional layers and parameters may improve expressivity, but they also increase optimization cost and can make a model harder to train. Ansatz selection must therefore balance expressivity, trainability, and available quantum resources.

On the practical side, quantum resources need to be considered too. How many qubits are available?  How many times does a circuit need to be run and sampled?  How long does this sampling take?  All of these questions are key to assessing the potential advantages of a QNN.


---
## 3. Building a Single-Qubit HQNN Workflow

We have covered classical NNs and QNNs which naturally brings us to the space in between, HQNNs.  Essentially an HQNN is some trainable model that contains a combination of quantum and classical layer(s). The exact composition of layers can be tuned for specific constraints such as the number of qubits available or the requirements for implementing a specific sort of technique. 

In this section you will explore construction of a HQNN to predict ocean surface temperature over a time series. That is, a model where a sequence of temperatures is input one at a time, and the model learns to predict the next temperature in the series. 

The workflow uses a classical **long short-term memory (LSTM)** model, a specialized neural network that extracts features from a window of earlier entries in a time series. The details of LSTM models are beyond the scope of this lesson; see the [PyTorch LSTM documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html) for the interface used here. For each input window, the LSTM and following linear layer produce two input-dependent rotation angles, $\theta_0$ and $\theta_1$, for $R_Y$ and $R_X$ on a single qubit (see the figure below). The quantum layer does not store separate trainable parameters. Instead, the parameter-shift rule computes derivatives with respect to these angles, and PyTorch uses the chain rule to update the trainable weights of the LSTM and linear layer.

![Hybrid architecture in which an LSTM produces two rotation angles for a single-qubit quantum layer](https://github.com/osbama/Phys710/blob/master/hands-on/hands-on-7-images/lstm_hqnn.png?raw=1)

In the previous QNN example, we converted a $Z_1$ expectation value into a class probability. Here, the raw expectation value of $Z$, $\bra{\psi}Z\ket{\psi}$, provides a continuous prediction between -1 and 1. The temperature data are min-max scaled approximately to the interval [0, 1], although held-out values may exceed that interval when they fall outside the training range. Most targets therefore lie within the quantum layer's output range, but targets above 1 cannot be represented exactly without an additional output transformation. Finally, the computed expectation value is compared to the true next temperature, and the resulting loss informs parameter updates in the LSTM model.


Most of the code is provided for you, but along the way, there will be exercises where you are asked to complete parts of the code.  

### 3.1 Loading the Data Set

The file **`ts_sea_temp_feat1_ds.npz`** contains a **sea surface temperature (SST) time series.** It has tens of thousands of samples. Each sample contains a short window of previous temperatures and a target — the **next** temperature after that window. The values are min-max scaled approximately to **[0, 1]** using the training range; held-out values can exceed this interval. The file provides separate training, validation, and test arrays. We preserve those supplied boundaries: training examples come only from the training array, validation examples only from the validation array, and test examples only from the test array.

To keep this instructional HQNN example fast while providing broad training coverage, the code samples 800 windows across the supplied training array and 100 windows across the validation array. The final 100 test windows are consecutive and remain in their original order. The final plot can therefore be read as a continuous sequence of next-temperature predictions over this test interval. The data file does not include timestamps or metadata describing how its three top-level split boundaries were selected, so the notebook preserves rather than redefines those supplied boundaries.

A data loader groups independent examples into **mini-batches**. This is standard practice for neural networks: a model processes several examples together, computes an average gradient for the batch, and makes better use of parallel hardware. Batching is especially important here because CUDA-Q can broadcast one `cudaq.observe` call over all circuit arguments in a batch. Each row still represents a separate circuit execution, but grouping the rows avoids thousands of individual Python and CUDA-Q dispatches. Shuffling the training loader changes which complete windows appear together; it does not change the chronological order of temperatures inside any individual window. Validation and test loaders are not shuffled. (See the training section below for a discussion of the validation set.)

In [ ]:
data = np.load("auxiliary_files/ts_sea_temp_feat1_ds.npz")
X_train_full, y_train_full = data["X_train"], data["y_train"]
X_valid_full, y_valid_full = data["X_valid"], data["y_valid"]
X_test_full, y_test_full = data["X_test"], data["y_test"]

# Preserve the supplied split boundaries. Sample only within the training
# and validation arrays, and keep a consecutive test block for plotting.
MAX_TRAIN_SAMPLES = 800
MAX_VALID_SAMPLES = 100
MAX_TEST_SAMPLES = 100

rng = np.random.default_rng(SEED)
train_indices = np.sort(
    rng.choice(len(y_train_full), MAX_TRAIN_SAMPLES, replace=False)
)
valid_indices = np.linspace(
    0, len(y_valid_full) - 1, MAX_VALID_SAMPLES, dtype=int
)

X_train, y_train = X_train_full[train_indices], y_train_full[train_indices]
X_valid, y_valid = X_valid_full[valid_indices], y_valid_full[valid_indices]
X_test = X_test_full[-MAX_TEST_SAMPLES:]
y_test = y_test_full[-MAX_TEST_SAMPLES:]
print(
    f"HQNN data: train={len(y_train)}, valid={len(y_valid)}, "
    f"test={len(y_test)}"
)

BATCH_SIZE = 128

def to_loader(X, y, features=1, batch_size=BATCH_SIZE, shuffle=False):
    if X.ndim == 2:
        X = X.reshape(X.shape[0], X.shape[1], features)  # (N, T, F)
    X_t = torch.from_numpy(X).float()
    y_t = torch.from_numpy(y).float()
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_train, y_train, shuffle=True)
valid_loader = to_loader(X_valid, y_valid, shuffle=False)
test_loader  = to_loader(X_test,  y_test, shuffle=False)

### 3.2 Building the Model

PyTorch models are commonly built by composing components derived from `nn.Module`. Each component defines how inputs produce outputs through a `forward` method. In the earlier classical example, `nn.Sequential` supplied this behavior by applying its listed layers in order, so you did not need to write `forward` yourself.

For this lesson, assume the quantum circuit may execute on a quantum device and is therefore a black box to PyTorch. Classical backpropagation cannot inspect the internal quantum state or trace operations performed by that device. PyTorch builds a computation graph by recording tensor operations as they execute; a call to an external quantum simulator breaks that graph because the simulator's internal operations are invisible to PyTorch's autograd engine. We instead define a custom PyTorch autograd function: its `forward` method obtains expectation values from CUDA-Q, and its `backward` method obtains the required quantum derivatives from additional parameter-shift circuit evaluations.

PyTorch uses those supplied derivatives to continue the chain rule through the surrounding classical layers. The quantum device is not performing backpropagation; it is only evaluating the original and parameter-shifted circuits requested by the classical training workflow.

The shift of $\delta = \pi/2$ is not arbitrary. For parameterized rotation gates with Pauli generators — such as the $R_Y$, $R_X$, and $R_Z$ gates used here — expectation values are exactly sinusoidal functions of the rotation angle. This means the derivative at any point equals the exact finite difference between evaluations at $\theta + \pi/2$ and $\theta - \pi/2$, divided by 2. This is not an approximation like a classical finite-difference gradient — it is the exact derivative, made possible by the known periodicity of Pauli rotation gates. Gates with more complex generators require generalized parameter-shift rules with different shift values.




**Exercise 3:**

Complete the code where specified to build the `QuantumFunction` class for our HQNN.


**a.** The `QuantumFunction` takes the number of qubits and Hamiltonian we are using as inputs, and defines the key information necessary for the layer. Find where the CUDA-Q kernel is defined and enter the proper rotation gates.


**b.** Next, define a function called `run` to execute the quantum circuit. Modify the function to compute the expectation values and save them as `results`. `theta_vals` contains one row of circuit parameters for each example in the current mini-batch. The `run` method submits these parameter rows together and returns one expectation value per row. Hint: remember that you are working within a class when you enter your variables.


**c.** The `forward` method runs the quantum circuit and returns its expectation values to the next model layer. It also saves the circuit parameters and other information on `ctx` so they are available during the later `backward` call. Complete the return statement so that `forward` returns the expectation values.


**d.** Ordinary classical backpropagation cannot propagate through the black-box quantum-device execution. Instead, use the **parameter-shift rule** to obtain the derivative of the measured expectation value. For each parameter, evaluate the circuit at $\theta+\delta$ and $\theta-\delta$, where $\delta=\pi/2$, and compute $(\bra{\psi(\theta+\delta)}Z\ket{\psi(\theta+\delta)}-\bra{\psi(\theta-\delta)}Z\ket{\psi(\theta-\delta)})/2$. These derivatives allow PyTorch to continue backpropagation through the surrounding classical layers without accessing the internal quantum state. Complete the `backward` method using these parameter-shift evaluations.



In [ ]:
class QuantumFunction(Function):
    """Allows the quantum circuit to input data, output expectation values
    and calculate derivatives with respect to input circuit angles via the parameter-shift rule"""

    def __init__(self, qubit_count: int, hamiltonian: cudaq.SpinOperator):
        """Define the quantum circuit in CUDA Quantum"""

        @cudaq.kernel
        def kernel(qubit_count: int, thetas: list[float]):

            qubits = cudaq.qvector(qubit_count)

            # ##TODO## Apply the two parameterized rotation gates.

        self.kernel = kernel
        self.qubit_count = qubit_count
        self.hamiltonian = hamiltonian

    def run(self, theta_vals: torch.Tensor) -> torch.Tensor:
        """Execute the quantum circuit to output an expectation value"""

        qubit_count = [self.qubit_count for _ in range(theta_vals.shape[0])]

        # ##TODO## Execute the batched kernel with cudaq.observe.
        raise NotImplementedError("Complete QuantumFunction.run")

        exp_vals = [results[i].expectation() for i in range(len(results))]
        exp_vals = torch.tensor(
            exp_vals, device=theta_vals.device, dtype=theta_vals.dtype
        )

        return exp_vals

    @staticmethod
    def forward(ctx, thetas: torch.Tensor, quantum_circuit,
                shift) -> torch.Tensor:

        # Save shift and quantum_circuit in context to use in backward.
        ctx.shift = shift
        ctx.quantum_circuit = quantum_circuit

        # Calculate expectation value.
        exp_vals = ctx.quantum_circuit.run(thetas).reshape(-1, 1)

        ctx.save_for_backward(thetas)

        # ##TODO## Return the values needed by the next model layer.
        return exp_vals  # TODO: confirm this is the correct tensor to return

    @staticmethod
    def backward(ctx, grad_output):
        """Backward pass computation via the parameter shift rule"""

        (thetas,) = ctx.saved_tensors

        gradients = torch.zeros_like(thetas)

        for i in range(thetas.shape[1]):

            thetas_plus = thetas.clone()
            thetas_plus[:, i] += 0.0  # ##TODO## Apply the positive shift.
            # TODO: compute exp_vals_plus by evaluating the circuit with theta shifted by +pi/2
            # (This line is intentionally None until you complete the exercise — you will see a TypeError if you run it as-is)
            exp_vals_plus = None  # ##TODO## Evaluate the shifted circuit.

            thetas_minus = thetas.clone()
            thetas_minus[:, i] -= 0.0  # ##TODO## Apply the negative shift.
            exp_vals_minus = None  # ##TODO## Evaluate the shifted circuit.

            gradients[:, i] = (exp_vals_plus - exp_vals_minus) / 2.0

        gradients = torch.mul(grad_output, gradients)

        return gradients, None, None


**Verification exercise: Check the parameter-shift gradient before training<**

Before running the full training loop, verify that `QuantumFunction.backward` is correct on a minimal case.

1. Define a one-qubit kernel with a single `RY(theta)` gate measuring `<Z>`.
2. Compute the gradient using your parameter-shift implementation: `grad = (f(theta + π/2) - f(theta - π/2)) / 2`.
3. Compute a finite-difference approximation: `grad_fd = (f(theta + ε) - f(theta - ε)) / (2ε)` for a small `ε` (e.g. `1e-4`).
4. Assert `|grad - grad_fd| < 1e-4`.

A wrong sign or shift value in the parameter-shift rule can produce gradients that look plausible — the loss decreases slowly or oscillates — while being systematically incorrect. This check catches that before training begins.



The cell below defines one more function outside of the PyTorch workflow to compute the statistics for the loss function like the MSE. 

In [ ]:
def metrics(y_true: torch.Tensor, y_pred: torch.Tensor):
    """
    y_true, y_pred: 1D tensors on SAME device (cpu or cuda)
    returns: mse, rmse, r2 as Python floats
    """
    # ensure 1D, same device
    device = y_true.device
    y_true = y_true.view(-1).to(device)
    y_pred = y_pred.view(-1).to(device)

    mse_t = torch.mean((y_true - y_pred) ** 2)
    rmse_t = torch.sqrt(mse_t)

    ss_res = torch.sum((y_true - y_pred) ** 2)
    ss_tot = torch.sum((y_true - y_true.mean()) ** 2)
    r2_t = 1.0 - ss_res / ss_tot if ss_tot > 0 else torch.tensor(float("nan"), device=device)

    # convert to floats on CPU at the end
    mse = mse_t.detach().cpu().item()
    rmse = rmse_t.detach().cpu().item()
    r2 = r2_t.detach().cpu().item()
    return mse, rmse, r2

We can now define a `QuantumLayer` object derived from the standard `nn.Module` class. This wrapper lets us include the quantum operation in a model using normal PyTorch conventions. `QuantumLayer.forward` calls `QuantumFunction.apply`, which registers the custom quantum operation with PyTorch's autograd system. During the forward pass, PyTorch invokes `QuantumFunction.forward`. Later, when `loss.backward()` reaches this operation, PyTorch invokes `QuantumFunction.backward` and uses its parameter-shift derivatives. `QuantumLayer` therefore does not need to define its own `backward` method.

In [ ]:
class QuantumLayer(nn.Module):
    """Encapsulates a quantum circuit into a quantum layer that adheres to PyTorch convention"""

    def __init__(self, qubit_count: int, hamiltonian, shift: torch.Tensor):
        super(QuantumLayer, self).__init__()

        self.quantum_circuit = QuantumFunction(qubit_count, hamiltonian)
        self.register_buffer("shift", torch.as_tensor(shift))

    def forward(self, input):

        result = QuantumFunction.apply(input, self.quantum_circuit, self.shift)

        return result

The final construction is to define `Hybrid_QNN` and bring all the pieces together. First, an LSTM model is defined with a user specified number of hidden layer nodes (32 by default, but we will end up using 16). If you are interested in the details of `nn.LSTM`, consult the PyTorch docs [here](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html). An additional linear layer is defined to connect the hidden layer nodes to the two outputs corresponding to the quantum circuit parameters. Finally, the quantum layer is defined with the appropriate input parameters.


In [ ]:
qubit_count = 1
hamiltonian = spin.z(0)
shift = torch.tensor(torch.pi / 2)

class Hybrid_QNN(nn.Module):
    """Structure of the hybrid neural network with classical fully connected layers and quantum layers"""

    def __init__(self, input_size=1, hidden_size=32, num_layers=1, dropout=0.0):
        super(Hybrid_QNN, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 2)
        # The 2 outputs from the PyTorch fc layer feed into the 2 variational gates in the quantum circuit.
        self.quantum = QuantumLayer(qubit_count, hamiltonian, shift)
        # self.quantum = nn.Linear(2, 1)

    def forward(self, x):
        out, _ = self.lstm(x)        
        last = out[:, -1, :]          
        ann = self.fc(last)
        y_hat = self.quantum(ann) # applying the quantum layer
        return y_hat

### 3.3 Training and Testing the Model

The cell below creates a `train_model` function to train the model. Essentially, the function is provided with the model we built above, the previously defined loaders for the training and validation data sets, the number of training epochs, and the learning rate. You may be less familiar with the terms epoch and learning rate. An epoch is one complete pass through the training data. Additional epochs give the optimizer more opportunities to improve the model, but they do not guarantee better validation performance. Training can plateau or begin to overfit, so the number of epochs should be chosen by monitoring validation metrics as well as compute cost.

The learning rate determines how quickly parameters adjust. If the rate is too high, the optimization may overshoot useful parameter values; if it is too low, convergence may be impractically slow. Batch size also matters because it changes how many optimizer updates occur during each epoch. After increasing the batch size to reduce CUDA-Q dispatch overhead, we use a learning rate of $6\times10^{-3}$ and 30 epochs. Batching keeps this longer training run much faster than the original one-sample workflow.

You can explore the code below to see the PyTorch syntax for important steps like setting up the loss function, the optimizer, saving the training metrics, and other key tasks are performed.  Notice at this stage, this would be pretty universal to any PyTorch workflow, quantum or otherwise. Go back up to the basic workflow you used to predict points in the circle.  How many similarities do you see?

Notice that this model also includes a validation set, unlike our earlier example. The validation set is not used to update model parameters. Its metrics are evaluated after each epoch so you can monitor generalization and detect possible overfitting. If the training RMSE goes down while the validation RMSE goes up, the model may be fitting the training data too closely and generalizing poorly. This training function reports validation metrics but does not perform early stopping or restore the best-performing epoch; it returns the model from the final epoch.

Finally, an additional function (`test_model`) is defined to run the trained model on the test set to see how our HQNN performs.

In [ ]:
EPOCHS = 30
LEARNING_RATE = 6e-3

def train_model(
    model, train_loader, valid_loader, epochs,
    learning_rate=LEARNING_RATE
):
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    train_history = {"mse": [], "rmse": [], "r2": []}
    val_history   = {"mse": [], "rmse": [], "r2": []}
    
    for epoch in range(1, epochs + 1):
        # ---- TRAIN ----
        model.train()
    
        for xb, yb in tqdm(train_loader):
            xb, yb = xb.to(device), yb.to(device)
    
            optimizer.zero_grad()
            pred = model(xb).squeeze(-1)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
    
    
        # Evaluate the completed epoch once so every reported training
        # prediction comes from the same version of the model.
        model.eval()
        train_true_all, train_pred_all = [], []
        with torch.no_grad():
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).squeeze(-1)
                train_true_all.append(yb)
                train_pred_all.append(pred)

        y_true = torch.cat(train_true_all).to(device)
        y_pred = torch.cat(train_pred_all).to(device)
        train_mse, train_rmse, train_r2 = metrics(y_true, y_pred)
    
        # ---- VALIDATION ----
        model.eval()
        val_true_all, val_pred_all = [], []
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).squeeze(-1)
                val_true_all.append(yb)
                val_pred_all.append(pred)
    
        val_true = torch.cat(val_true_all).to(device)
        val_pred = torch.cat(val_pred_all).to(device)
        val_mse, val_rmse, val_r2 = metrics(val_true, val_pred)
    
        train_history["mse"].append(train_mse)
        train_history["rmse"].append(train_rmse)
        train_history["r2"].append(train_r2)
        val_history["mse"].append(val_mse)
        val_history["rmse"].append(val_rmse)
        val_history["r2"].append(val_r2)
    
        print(
            f"Epoch {epoch:03d} | "
            f"train MSE: {train_mse:.4f} | RMSE: {train_rmse:.4f} | R2: {train_r2:.4f} \n"
            f"val MSE: {val_mse:.4f} | RMSE: {val_rmse:.4f} | R2: {val_r2:.4f}"
        )
        print("-"*100)

    return model, train_history, val_history

def test_model(model, test_loader):
    model.eval()
    
    test_true_all, test_pred_all = [], []
    
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).squeeze(-1)
            test_true_all.append(yb)
            test_pred_all.append(pred)
    
    test_true = torch.cat(test_true_all).to(device)
    test_pred = torch.cat(test_pred_all).to(device)
    
    test_mse, test_rmse, test_r2 = metrics(test_true, test_pred)
    print(
        f"Test MSE: {test_mse:.6f} | "
        f"Test RMSE: {test_rmse:.4f} | "
        f"Test R2: {test_r2:.4f}"
    )

    return test_mse, test_rmse, test_r2, test_true, test_pred

Now, set the model to be our HQNN and train it! You might have also noticed the `.to(device)` commands in this line and the cells above. These keep the LSTM, its parameters, and its input tensors on the GPU. Quantum expectation values still cross the CUDA-Q–PyTorch boundary, so batching is important: it greatly reduces the number of separate runtime dispatches and data transfers.

The plots generated compare the metrics for the training and validation sets.  How do they look?  Did the model train?

In [ ]:
# Reset the PyTorch random state here so rerunning Exercise 3 starts from
# the same model parameters and uses the same shuffled mini-batches.
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

model = Hybrid_QNN(input_size=1, hidden_size=16).to(device)
print(
    f"Training configuration: batch size={BATCH_SIZE}, epochs={EPOCHS}, "
    f"learning rate={LEARNING_RATE}, seed={SEED}"
)

final_model, train_history, val_history = train_model(
    model, train_loader, valid_loader, epochs=EPOCHS,
    learning_rate=LEARNING_RATE
)

def plot_history(train_history, val_history, min_mark=0):
    epoch_range = np.arange(len((train_history['mse'])))
    
    # NVIDIA green (train) and purple (val) on dark background
    TRAIN_COLOR, VAL_COLOR = "#76b900", "#A020F0"
    plt.style.use("dark_background")
    fig, ax = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
    fig.patch.set_facecolor("#1A1A1A")
    for a in ax:
        a.set_facecolor("#1A1A1A")
        a.tick_params(colors="#E0E0E0")
        a.spines["bottom"].set_color("#444444")
        a.spines["top"].set_color("#444444")
        a.spines["left"].set_color("#444444")
        a.spines["right"].set_color("#444444")
        a.yaxis.label.set_color("#E0E0E0")
        a.xaxis.label.set_color("#E0E0E0")
    ax[0].plot(epoch_range[min_mark:], train_history["r2"][min_mark:], label="train", color=TRAIN_COLOR, linewidth=2)
    ax[0].plot(epoch_range[min_mark:], val_history["r2"][min_mark:], label="val", color=VAL_COLOR, linewidth=2)
    
    ax[1].plot(epoch_range[min_mark:], train_history["mse"][min_mark:], label="train", color=TRAIN_COLOR, linewidth=2)
    ax[1].plot(epoch_range[min_mark:], val_history["mse"][min_mark:], label="val", color=VAL_COLOR, linewidth=2)
    
    ax[2].plot(epoch_range[min_mark:], train_history["rmse"][min_mark:], label="train", color=TRAIN_COLOR, linewidth=2)
    ax[2].plot(epoch_range[min_mark:], val_history["rmse"][min_mark:], label="val", color=VAL_COLOR, linewidth=2)
    
    ax[2].set_xlabel("Epoch")
    
    ax[0].set_ylabel("R2")
    ax[1].set_ylabel("MSE")
    ax[2].set_ylabel("RMSE")
    
    for a in ax:
        a.grid(True, color="#444444", linestyle="--", alpha=0.6)
    ax[0].legend(facecolor="#2A2A2A", edgecolor="#444444", labelcolor=[TRAIN_COLOR, VAL_COLOR])
    ax[1].legend(facecolor="#2A2A2A", edgecolor="#444444", labelcolor=[TRAIN_COLOR, VAL_COLOR])
    ax[2].legend(facecolor="#2A2A2A", edgecolor="#444444", labelcolor=[TRAIN_COLOR, VAL_COLOR])
    plt.show()

plot_history(train_history, val_history, min_mark=0)

Run the trained model on the later, unseen test set. How did it do?

In [ ]:
mse, rmse, r2, ytrue, ypred = test_model(final_model, test_loader)

Finally, we can unscale the data and plot the predicted values against the true values. How do they look?

In [ ]:
def minmax_inverse_scale(x_scaled, data_min=15.473, data_max=28.258):
    return x_scaled * (data_max - data_min) + data_min

# Prepare data
T = np.arange(ytrue.size()[0])
gt = minmax_inverse_scale(ytrue.detach().cpu().reshape(-1,1))
preds = minmax_inverse_scale(ypred.detach().cpu().reshape(-1,1))


plt.style.use('dark_background') # Helper to set general defaults to dark
fig, ax = plt.subplots(1, 1, figsize=(11, 4))
fig.patch.set_facecolor('black')
ax.set_facecolor('black')


ax.plot(T, gt, color='white', linewidth=1, label='Original')
ax.plot(T, preds, color='#76b900', linewidth=1, label='Predicted (HQNN)')


ax.set_ylabel('Temperature [°C]', color='white', fontsize=12)
ax.set_xlabel('Test time step', color='white', fontsize=12)
ax.set_title('Sea Surface Temperature Forecasting', color='white', fontsize=14, pad=15)


ax.tick_params(axis='x', colors='white')
ax.tick_params(axis='y', colors='white')
for spine in ax.spines.values():
    spine.set_color('white')

ax.grid(color='#444444', linestyle='--', linewidth=0.5, alpha=0.5)

legend = ax.legend(facecolor='black', edgecolor='white', framealpha=1)
plt.setp(legend.get_texts(), color='white')

plt.show()

---
## Conclusion

### Key Takeaways

* Balanced training data are essential for interpreting classifier accuracy; a high score can otherwise reflect class imbalance rather than a learned decision boundary.
* A QNN combines data encoding, a parameterized ansatz, and an observable readout. Entangling gates can allow multiple encoded features to influence a shared readout, but they do not by themselves guarantee better accuracy or a quantum advantage.
* CUDA-Q expectation values can be incorporated into PyTorch through a custom autograd function. Parameter-shift circuit evaluations provide derivatives with respect to the input rotation angles, and PyTorch uses those derivatives to update the surrounding classical network.
* In the forecasting HQNN, the LSTM and linear layer generate input-dependent quantum rotation angles; the quantum layer does not store separate trainable parameters.
* This small, simulator-based workflow demonstrates how to construct and train an HQNN, not evidence of a practical quantum advantage. Larger studies must also account for data-encoding cost, trainability, hardware noise, runtime, and strong classical baselines.

### Agentic AI Extension: Optimize the Exercise 2 QNN

Use an agentic **autoresearch loop** to find the best classical optimizer tested for the QNN in Exercise 2. Keep the circuit, training data, and initial parameters fixed while the agent tries routines such as COBYLA, Nelder–Mead, and Powell. Give every optimizer the same budget of quantum circuit executions; because one call to `cost_function` evaluates the circuit once for every training point, count the number of cost-function calls rather than relying on each optimizer's definition of an iteration. Have the agent record the final training MSE, select the best result under the budget, and evaluate that choice on the test set once.

**Next steps:** Review parameterized kernels and variational optimization in the Quick Start track, then compare this HQNN workflow with other hybrid quantum–classical applications.

# Advanced Hybrid Quantum Neural Networks — Hybrid Workflows: Performance and Scaling
$\renewcommand{\ket}[1]{|#1\rangle}\renewcommand{\bra}[1]{\langle#1|}$

---

**What You Will Do:**
* Enable cuDNN for a PyTorch LSTM and interpret controlled component timings
* Compare batch sizes and learning rates using runtime and validation RMSE
* Implement an SPSA quantum gradient with optional MQPU dispatch
* Extend a one-output HQNN to a three-qubit, three-output forecast
* Direct an AI coding agent to design and evaluate a scaled CUDA-Q HQNN

**Prerequisites:**
* Python, Jupyter, NumPy, and introductory PyTorch training loops
* Qubits, parameterized gates, expectation values, and variational circuits

**Key Terminology:**
* Hybrid quantum neural network (HQNN)
* CUDA Deep Neural Network library (cuDNN)
* Mini-batch and learning rate
* Parameter-shift rule
* Simultaneous Perturbation Stochastic Approximation (SPSA)
* Multi-QPU (MQPU) execution
* Vector–Jacobian product

**CUDA-Q Syntax:**
* [`@cudaq.kernel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.kernel) — defines a quantum kernel
* [`cudaq.observe`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.observe) — evaluates expectation values
* [`cudaq.observe_async`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.observe_async) — submits an asynchronous expectation-value job
* [`cudaq.set_target`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.set_target) — selects a simulator or hardware target




> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).


In [ ]:
# Standard library
import copy
import time

# Scientific computing and visualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Machine learning
import torch
import torch.nn as nn
from torch.autograd import Function
from torch.utils.data import DataLoader, TensorDataset

# Notebook display
from IPython.display import display

# CUDA-Q
import cudaq
from cudaq import spin

SEED = 111
torch.manual_seed(SEED)
np.random.seed(SEED)
cudaq.set_random_seed(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
cudaq.set_target("nvidia" if torch.cuda.is_available() else "qpp-cpu")
QUICK_RUN = device.type != "cuda"
print(f"PyTorch device: {device}")
print(f"CUDA-Q target: {cudaq.get_target().name}")
print(f"cuDNN available: {torch.backends.cudnn.is_available()}")
print(f"Quick-run mode: {QUICK_RUN}")
if QUICK_RUN:
    print("CPU detected: GPU-only timing and training experiments will be skipped.")
else:
    print("GPU detected: quantum-gradient batches can take a while; progress bars show that work is active.")

---
## 1. Introduction

A **hybrid quantum neural network (HQNN)** connects a classical neural network to a parameterized quantum circuit and trains the combined computation through a shared loss. This lesson retains the sea-surface-temperature HQNN from [Part 1](01_an_introduction_to_hybrid_quantum_neural_networks.ipynb) and asks how to make its training workflow more efficient and extensible without hiding the relevant tradeoffs.

The **CUDA Deep Neural Network library (cuDNN)** supplies optimized GPU kernels for the classical recurrent layer. A **mini-batch** groups examples into one loss and gradient calculation, while the **learning rate** controls the overall optimizer step size. The **parameter-shift rule** supplies exact circuit derivatives but uses two shifted evaluations for every circuit parameter. **Simultaneous Perturbation Stochastic Approximation (SPSA)** instead estimates a gradient from two simultaneous perturbations, and **multi-QPU (MQPU) execution** can dispatch independent quantum jobs across available QPUs. For several circuit outputs, PyTorch combines their derivatives through a **vector–Jacobian product**.

The examples are intentionally small enough to run during a lesson. They demonstrate controlled measurements and implementation patterns; they do not claim quantum advantage or state-of-the-art forecasting accuracy.

---
## 2. Begin with the Part 1 Forecasting Model

Part 1 supplied a time series that had already been split, scaled, and windowed with `lookback=2` and `horizon=1`. The first portion of the next cell preserves those split boundaries, selects 800 training windows, 100 validation windows, and the final 100 consecutive test windows, and creates the same data loaders. The data file is stored beside this notebook.

For the lesson, treat this as the same baseline HQNN from Part 1. After the marked Part 2 boundary, two new utility functions reconstruct and re-window each split. We will use them later for the larger cuDNN timing probe and the three-step forecasting task. On a CPU-only system, `QUICK_RUN` skips the dispatch-heavy training comparisons, but it does not alter the Part 1 baseline data.

### The reusable single-qubit HQNN

The single code cell below contains the complete reusable setup from Part 1: data preparation, the HQNN classes, metrics, and training and test helpers. Run it once from top to bottom before continuing so that no baseline preparation cell is missed. The class names, tensor shapes, LSTM, linear layer, circuit, and parameter-shift calculation remain the same. Minor packaging and experiment-control additions begin only at the comments marked `Part 2-only`.

The LSTM and linear layer generate two input-dependent angles, the quantum layer applies `ry` and `rx`, and the model returns the expectation value of `Z`. The quantum layer has no independent trainable weights; its derivatives with respect to the two angles allow PyTorch to continue the chain rule into the linear layer and LSTM.

In [ ]:
data = np.load("auxiliary_files/ts_sea_temp_feat1_ds.npz")
X_train_full, y_train_full = data["X_train"], data["y_train"]
X_valid_full, y_valid_full = data["X_valid"], data["y_valid"]
X_test_full, y_test_full = data["X_test"], data["y_test"]

# Preserve the supplied split boundaries. Sample only within the training
# and validation arrays, and keep a consecutive test block for plotting.
MAX_TRAIN_SAMPLES = 800
MAX_VALID_SAMPLES = 100
MAX_TEST_SAMPLES = 100

rng = np.random.default_rng(SEED)
train_indices = np.sort(
    rng.choice(len(y_train_full), MAX_TRAIN_SAMPLES, replace=False)
)
valid_indices = np.linspace(
    0, len(y_valid_full) - 1, MAX_VALID_SAMPLES, dtype=int
)

X_train, y_train = X_train_full[train_indices], y_train_full[train_indices]
X_valid, y_valid = X_valid_full[valid_indices], y_valid_full[valid_indices]
X_test = X_test_full[-MAX_TEST_SAMPLES:]
y_test = y_test_full[-MAX_TEST_SAMPLES:]
print(
    f"HQNN data: train={len(y_train)}, valid={len(y_valid)}, "
    f"test={len(y_test)}"
)

BATCH_SIZE = 128

def to_loader(X, y, features=1, batch_size=BATCH_SIZE, shuffle=False):
    if X.ndim == 2:
        X = X.reshape(X.shape[0], X.shape[1], features)  # (N, T, F)
    X_t = torch.from_numpy(X).float()
    y_t = torch.from_numpy(y).float()
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_train, y_train, shuffle=True)
valid_loader = to_loader(X_valid, y_valid, shuffle=False)
test_loader  = to_loader(X_test,  y_test, shuffle=False)

# Part 2-only utilities begin here.
def reconstruct_series(X, y):
    """Recover the ordered 1D series from lookback=2, horizon=1 windows."""
    return np.concatenate([X[0], y])

def make_windows(series, lookback=2, horizon=1):
    values = np.asarray(series, dtype=float)
    X, y = [], []
    for t in range(lookback, len(values) - horizon + 1):
        X.append(values[t - lookback:t])
        y.append(values[t:t + horizon])
    X = np.asarray(X)
    y = np.asarray(y)
    return X, y[:, 0] if horizon == 1 else y

train_series = reconstruct_series(X_train_full, y_train_full)
valid_series = reconstruct_series(X_valid_full, y_valid_full)
test_series = reconstruct_series(X_test_full, y_test_full)

# Part 1 HQNN model and training utilities

class QuantumFunction(Function):
    """Allows the quantum circuit to input data, output expectation values
    and calculate derivatives with respect to input circuit angles via the parameter-shift rule"""

    def __init__(self, qubit_count: int, hamiltonian: cudaq.SpinOperator):
        """Define the quantum circuit in CUDA Quantum"""

        @cudaq.kernel
        def kernel(qubit_count: int, thetas: np.ndarray):

            qubits = cudaq.qvector(qubit_count)

            ry(thetas[0], qubits[0])
            rx(thetas[1], qubits[0])

        self.kernel = kernel
        self.qubit_count = qubit_count
        self.hamiltonian = hamiltonian

    def run(self, theta_vals: torch.Tensor) -> torch.Tensor:
        """Execute the quantum circuit to output an expectation value"""

        qubit_count = [self.qubit_count for _ in range(theta_vals.shape[0])]

        results = cudaq.observe(
            self.kernel, self.hamiltonian, qubit_count, theta_vals
        )

        exp_vals = [results[i].expectation() for i in range(len(results))]
        exp_vals = torch.tensor(
            exp_vals, device=theta_vals.device, dtype=theta_vals.dtype
        )

        return exp_vals

    @staticmethod
    def forward(ctx, thetas: torch.Tensor, quantum_circuit,
                shift) -> torch.Tensor:

        # Save shift and quantum_circuit in context to use in backward.
        ctx.shift = shift
        ctx.quantum_circuit = quantum_circuit

        # Calculate expectation value.
        exp_vals = ctx.quantum_circuit.run(thetas).reshape(-1, 1)

        ctx.save_for_backward(thetas)

        return exp_vals

    @staticmethod
    def backward(ctx, grad_output):
        """Backward pass computation via the parameter shift rule"""

        (thetas,) = ctx.saved_tensors

        gradients = torch.zeros_like(thetas)

        for i in range(thetas.shape[1]):

            thetas_plus = thetas.clone()
            thetas_plus[:, i] += ctx.shift
            exp_vals_plus = ctx.quantum_circuit.run(thetas_plus)

            thetas_minus = thetas.clone()
            thetas_minus[:, i] -= ctx.shift
            exp_vals_minus = ctx.quantum_circuit.run(thetas_minus)

            gradients[:, i] = (exp_vals_plus - exp_vals_minus) / 2.0

        gradients = torch.mul(grad_output, gradients)

        return gradients, None, None


class QuantumLayer(nn.Module):
    """Encapsulates a quantum circuit into a quantum layer that adheres to PyTorch convention"""

    def __init__(self, qubit_count: int, hamiltonian, shift: torch.Tensor):
        super(QuantumLayer, self).__init__()

        self.quantum_circuit = QuantumFunction(qubit_count, hamiltonian)
        self.register_buffer("shift", torch.as_tensor(shift))

    def forward(self, input):

        result = QuantumFunction.apply(input, self.quantum_circuit, self.shift)

        return result


qubit_count = 1
hamiltonian = spin.z(0)
shift = torch.tensor(torch.pi / 2)

class Hybrid_QNN(nn.Module):
    """Structure of the hybrid neural network with classical fully connected layers and quantum layers"""

    def __init__(self, input_size=1, hidden_size=32, num_layers=1, dropout=0.0):
        super(Hybrid_QNN, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 2)
        # The 2 outputs from the PyTorch fc layer feed into the 2 variational gates in the quantum circuit.
        self.quantum = QuantumLayer(qubit_count, hamiltonian, shift)
        # self.quantum = nn.Linear(2, 1)

    def forward(self, x):
        out, _ = self.lstm(x)        
        last = out[:, -1, :]          
        ann = self.fc(last)
        y_hat = self.quantum(ann) # applying the quantum layer
        return y_hat

def metrics(y_true: torch.Tensor, y_pred: torch.Tensor):
    """
    y_true, y_pred: 1D tensors on SAME device (cpu or cuda)
    returns: mse, rmse, r2 as Python floats
    """
    # ensure 1D, same device
    device = y_true.device
    y_true = y_true.view(-1).to(device)
    y_pred = y_pred.view(-1).to(device)

    mse_t = torch.mean((y_true - y_pred) ** 2)
    rmse_t = torch.sqrt(mse_t)

    ss_res = torch.sum((y_true - y_pred) ** 2)
    ss_tot = torch.sum((y_true - y_true.mean()) ** 2)
    r2_t = 1.0 - ss_res / ss_tot if ss_tot > 0 else torch.tensor(float("nan"), device=device)

    # convert to floats on CPU at the end
    mse = mse_t.detach().cpu().item()
    rmse = rmse_t.detach().cpu().item()
    r2 = r2_t.detach().cpu().item()
    return mse, rmse, r2

EPOCHS = 30
LEARNING_RATE = 6e-3

def train_model(
    model, train_loader, valid_loader, epochs,
    learning_rate=LEARNING_RATE
):
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    train_history = {"mse": [], "rmse": [], "r2": []}
    val_history   = {"mse": [], "rmse": [], "r2": []}
    
    for epoch in range(1, epochs + 1):
        print(f"Starting epoch {epoch}/{epochs} ({len(train_loader)} batches)", flush=True)
        # ---- TRAIN ----
        model.train()
    
        for batch_index, (xb, yb) in enumerate(train_loader, start=1):
            if batch_index == 1 or batch_index % 10 == 0 or batch_index == len(train_loader):
                print(f"  training batch {batch_index}/{len(train_loader)} starting", flush=True)
            xb, yb = xb.to(device), yb.to(device)
    
            optimizer.zero_grad()
            pred = model(xb).squeeze(-1)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
    
    
        # Evaluate the completed epoch once so every reported training
        # prediction comes from the same version of the model.
        model.eval()
        train_true_all, train_pred_all = [], []
        with torch.no_grad():
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).squeeze(-1)
                train_true_all.append(yb)
                train_pred_all.append(pred)

        y_true = torch.cat(train_true_all).to(device)
        y_pred = torch.cat(train_pred_all).to(device)
        train_mse, train_rmse, train_r2 = metrics(y_true, y_pred)
    
        # ---- VALIDATION ----
        model.eval()
        val_true_all, val_pred_all = [], []
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).squeeze(-1)
                val_true_all.append(yb)
                val_pred_all.append(pred)
    
        val_true = torch.cat(val_true_all).to(device)
        val_pred = torch.cat(val_pred_all).to(device)
        val_mse, val_rmse, val_r2 = metrics(val_true, val_pred)
    
        train_history["mse"].append(train_mse)
        train_history["rmse"].append(train_rmse)
        train_history["r2"].append(train_r2)
        val_history["mse"].append(val_mse)
        val_history["rmse"].append(val_rmse)
        val_history["r2"].append(val_r2)
    
        print(
            f"Epoch {epoch:03d} | "
            f"train MSE: {train_mse:.4f} | RMSE: {train_rmse:.4f} | R2: {train_r2:.4f} \n"
            f"val MSE: {val_mse:.4f} | RMSE: {val_rmse:.4f} | R2: {val_r2:.4f}"
        )
        print("-"*100)

    return model, train_history, val_history

def test_model(model, test_loader):
    model.eval()
    
    test_true_all, test_pred_all = [], []
    
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).squeeze(-1)
            test_true_all.append(yb)
            test_pred_all.append(pred)
    
    test_true = torch.cat(test_true_all).to(device)
    test_pred = torch.cat(test_pred_all).to(device)
    
    test_mse, test_rmse, test_r2 = metrics(test_true, test_pred)
    print(
        f"Test MSE: {test_mse:.6f} | "
        f"Test RMSE: {test_rmse:.4f} | "
        f"Test R2: {test_r2:.4f}"
    )

    return test_mse, test_rmse, test_r2, test_true, test_pred

# Part 2-only timing helper begins here.
def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize()

---
## 3. Accelerate the classical layer with cuDNN

[NVIDIA cuDNN](https://docs.nvidia.com/deeplearning/cudnn/latest/)—the **CUDA Deep Neural Network library**—provides highly optimized GPU implementations of operations used repeatedly in deep learning. Common examples include convolutions, matrix multiplication, pooling, normalization, attention, pointwise operations, and recurrent networks such as RNNs and LSTMs. Frameworks such as PyTorch call these kernels behind the scenes rather than requiring us to program them directly. 

### Where cuDNN fits in this HQNN

Your workflow contains two distinct GPU workloads:

- PyTorch executes the classical `nn.LSTM` and linear layer.
- CUDA-Q executes the quantum circuit.

CUDA is the general GPU-computing platform. **cuDNN supplies specialized deep-learning kernels that run on a CUDA GPU.** In this lesson, cuDNN can accelerate the PyTorch LSTM, but it does not execute or modify the CUDA-Q quantum circuit which uses its own optimized CUDA kernels for quantum circuit simulation.

The `Hybrid_QNN` class does **not** need a new LSTM or a new `forward` method. The Part 1 code already does the two essential setup steps: it creates a CUDA device and moves both the model and each mini-batch to that device. We only make the cuDNN choice explicit before constructing and running the model:

```python
torch.backends.cudnn.enabled = True
model = Hybrid_QNN(input_size=1, hidden_size=16).to(device)
xb = xb.to(device)
```

PyTorch normally enables cuDNN automatically when it is installed and the LSTM inputs are supported. Writing the flag explicitly documents our intent and lets us create a controlled comparison. Setting it to `False` does **not** move the LSTM to the CPU: it keeps the same CUDA tensors on the same GPU and selects a non-cuDNN CUDA path. Therefore, the comparison below changes the implementation backend, not the model architecture.

![Where cuDNN fits in the Part 1 HQNN workflow](https://github.com/osbama/Phys710/blob/master/hands-on/hands-on-7-images/cudnn_hqnn_workflow.png?raw=1)



**Exercise 1: Compare the HQNN LSTM with and without cuDNN**

The next two code cells provide the baseline and scaled comparisons. You do not need to edit the Part 1 implementation or rewrite `Hybrid_QNN`. The `TODO` comments identify the settings that turn cuDNN off and on.

**a.** Run the baseline comparison. Confirm that it uses LSTM settings of `lookback=2`, `hidden_size=16`, a one-layer LSTM, and reports timings with cuDNN off and on.

**b.** Confirm that the unchanged HQNN produces shape `(batch, 1)`. Explain why toggling `torch.backends.cudnn.enabled` requires no change to `nn.LSTM`, the linear layer, or the CUDA-Q circuit.

**c.** In the next cell, run the scaled comparison with `lookback=64`, `hidden_size=64`, two layers, and `batch_size=256`. Compare its speedup with the baseline result.

**d.** Change one field in `SCALED_LSTM`, rerun the cell, and record both milliseconds and speedup. A configuration may improve, remain unchanged, or slow down.

**e.** Explain why an isolated LSTM speedup may not produce the same end-to-end HQNN speedup when CUDA-Q execution and quantum-gradient evaluations dominate runtime.



In [ ]:
# Exercise 1 — select the two cuDNN settings to compare.
BASELINE_CUDNN_SETTINGS = [
    # TODO: add the cuDNN-off setting
    # TODO: add the cuDNN-on setting
]

# The timing and reporting code is available in the solution notebook.
raise NotImplementedError("Complete Exercise 1 before running this cell.")


> **Interpretation:** The baseline LSTM may show little speedup—or even a slowdown—because it processes only two timesteps with 16 hidden units. At that scale, launch and framework overhead can outweigh the benefit of an optimized recurrent kernel. Treat a result near $1\times$ as inconclusive rather than as evidence that cuDNN is ineffective; use the larger workload below for a clearer component-level comparison.



The Part 1 LSTM processes only two timesteps with 16 hidden units. That small amount of recurrent work may finish before an optimized backend can overcome its own launch overhead. For a clearer component-level demonstration, we will use the same temperature series with `lookback=64`, `hidden_size=64`, two recurrent layers, and `batch_size=256`. This shape showed a visible cuDNN benefit on the development GPU while remaining small enough for the lesson.

The benchmark times one warmed-up **forward pass** with cuDNN disabled and enabled. Both cases receive the same input tensor and seeded initial weights on the same CUDA GPU. Ten untimed warm-up passes allow kernel setup to finish, and CUDA events avoid timing only an asynchronous kernel launch. The final linear and CUDA-Q layers are deliberately excluded so the classical LSTM effect remains visible.

In [ ]:
# Exercise 1 — configure the scaled LSTM comparison.
SCALED_LSTM = {
    "workload": "Scaled LSTM",
    "lookback": 64,
    "hidden_size": 64,
    "num_layers": 2,
    "batch_size": 256,
}
SCALED_CUDNN_SETTINGS = [
    # TODO: add the cuDNN-off setting
    # TODO: add the cuDNN-on setting
]

raise NotImplementedError("Complete the scaled cuDNN comparison.")



cuDNN performance depends on the GPU, software versions, tensor shapes, precision, and selected algorithms, so speedups vary. Larger compute-bound recurrent workloads generally give optimized kernels more opportunity to amortize overhead.

This change accelerates only the classical portion of the HQNN. The quantum portion still includes CUDA-Q circuit-execution overhead, whether circuits are simulated on a GPU or submitted to a QPU. End-to-end performance therefore depends on which component dominates the workload.

For a published analysis of a substantially larger solar-irradiance pipeline that combines cuDNN acceleration for classical neural-network work with CUDA-Q for quantum execution, see [*Solar Irradiance Forecasting Using a Hybrid Quantum Neural Network: A Comparison on GPU-Based Workflow Development Platforms*](https://doi.org/10.1109/ACCESS.2024.3472053). NVIDIA's [technical overview](https://developer.nvidia.com/blog/accelerating-quantum-algorithms-for-solar-energy-prediction-with-nvidia-cuda-q-and-nvidia-cudnn/) provides an accessible visual summary of that end-to-end workflow.

---
## 4. Batching and learning rate

Improving an HQNN is not only a question of changing its architecture. We also need to organize each epoch so that the available compute is used efficiently without losing useful optimization behavior. This section focuses on two familiar training choices—batch size and learning rate—that were used in Part 1 but not examined there in detail.

Suppose the training set contains $N$ examples. One **epoch** means that the model processes all $N$ examples once. There are several ways to organize that work. The **batch size** $B$ determines how many examples are grouped into one training round; it does not change the number of examples in the epoch.

Every training round performs the same four actions:

1. Run $B$ examples through the HQNN to obtain predictions.
2. Combine their errors into one mini-batch loss. Here, `nn.MSELoss` averages the squared errors.
3. Compute the gradient of that loss with respect to the model parameters.
4. Use the optimizer to update the parameters once.

The number of optimizer updates in one epoch is therefore

$$\text{updates per epoch}=\left\lceil\frac{N}{B}\right\rceil.$$

- With $B=1$, one example produces one gradient and one update, so the epoch has $N$ short rounds.
- With $1<B<N$, each gradient summarizes a mini-batch and the epoch has fewer, larger rounds.
- With $B=N$, the entire epoch produces one loss, one gradient, and one update. This is full-batch training.

<img src="https://github.com/osbama/Phys710/blob/master/hands-on/hands-on-7-images/batching_cudaq_calls.svg?raw=1" alt="Diagram showing one epoch processed as single-example rounds or larger mini-batch rounds" width="100%"/>

There are important tradeoffs in how batching changes training behavior. For this HQNN, the main performance benefit is fewer separate gradient-and-update rounds, allowing the logical quantum evaluations to be grouped into larger batches. Batching does not remove the underlying parameter-shift work across the epoch.

A gradient is computed in **every** training round. PyTorch can differentiate the LSTM and linear layer directly, but it cannot trace through execution on a quantum simulator or QPU. The current `QuantumFunction.backward` therefore supplies the quantum derivative using the parameter-shift rule introduced in [Part 1](01_an_introduction_to_hybrid_quantum_neural_networks.ipynb):

$$\frac{\partial f}{\partial \theta_i}=\frac{f(\theta_i+\pi/2)-f(\theta_i-\pi/2)}{2}.$$

For $P$ circuit parameters, parameter shift requires **$2P$ shifted logical circuit evaluations per example**—one positive and one negative shift for every parameter. The current circuit has $P=2$ angles, so its quantum gradient requires **four shifted circuit evaluations per example**. Including the unshifted forward evaluation, that is five logical circuit evaluations per example in one training round. A mini-batch of $B$ examples therefore represents $4B$ shifted evaluations during the backward pass plus $B$ forward evaluations.

CUDA-Q can process many parameter rows together, which reduces repeated Python and submission overhead and lets suitable backends exploit parallel work. Batching does not change the parameter-shift rule itself: each parameter still needs its positive and negative shifts. This is why both the number of training rounds and the cost of the gradient inside each round matter for runtime.

When the batch size changes substantially, the learning rate often needs to be reconsidered rather than copied over automatically.

The **learning rate** $\eta$ controls the size of each optimizer update. Ordinary gradient descent can be summarized as

$$\text{parameters}\leftarrow\text{parameters}-\eta\,\text{gradient}.$$

This notebook uses Adam, which adapts and rescales the gradient, but `learning_rate` still sets the overall step-size scale. If it is too small, the model may improve only slowly. If it is too large, updates can overshoot useful parameter values and validation RMSE may oscillate or worsen.

Batch size and learning rate must therefore be interpreted together. A large batch gives fewer updates per epoch, so a learning rate that worked with many small-batch updates may move the model too little under the same epoch budget. Increasing it may help, but scaling the learning rate with batch size is only a starting hypothesis—not a guarantee. The experiments below separate these effects: Part A fixes the learning rate and compares one-epoch timing, while Part B fixes the batch size and compares learning-rate trajectories over the same number of updates.



**Exercise 2: Batch size, runtime, and learning rate**

The next two code cells perform HQNN training with the same initial weights and 128-example training subset.

**a.** Run Part A with `batch_size=1`, `64`, and `128`. Compare the number of training rounds and elapsed time. Why is `batch_size=1` usually slow in this hybrid workflow?

**b.** Compare the validation RMSE values. Explain why they are useful observations but not a fair accuracy comparison: how many updates does each configuration receive in one epoch?

**c.** Run Part B with `batch_size=128`. Use the validation-RMSE curves to identify a rate that changes too slowly, one that learns usefully, and—if present—one that overshoots.

**d.** Try one additional batch size or learning rate. Describe the result without assuming that a larger value is always better.

</div>

### Part A — time one real training epoch

The following cell performs the full forward, loss, gradient, and optimizer-update sequence for every mini-batch. The learning rate and number of training examples stay fixed; only `batch_size` changes. The timer covers the training epoch itself, not validation or testing.

> **Runtime note:** The unbatched case intentionally performs 128 separate updates and took about one minute on the development GPU. Exact times depend on the hardware and software environment.

In [ ]:
BATCH_TIMING_TRAIN_SAMPLES = 128
BATCH_TIMING_LEARNING_RATE = 6e-3
BATCH_SIZES = (1, 64, 128)

def time_one_training_epoch(batch_size):
    configure_cudnn(True)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    train_loader, valid_loader, _ = single_step_loaders(
        batch_size, train_samples=BATCH_TIMING_TRAIN_SAMPLES
    )
    model = Hybrid_QNN(input_size=1, hidden_size=16).to(device)
    model.load_state_dict(SINGLE_QUBIT_INITIAL_STATE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=BATCH_TIMING_LEARNING_RATE
    )

    model.train()
    synchronize()
    start = time.perf_counter()
    print(f"Timing batch_size={batch_size}: {len(train_loader)} batches", flush=True)
    for batch_index, (xb, yb) in enumerate(train_loader, start=1):
        if batch_index == 1 or batch_index % 10 == 0 or batch_index == len(train_loader):
            print(f"  timing batch {batch_index}/{len(train_loader)} starting", flush=True)
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        prediction = model(xb).squeeze(-1)
        loss = criterion(prediction, yb)
        loss.backward()
        optimizer.step()
    synchronize()
    elapsed = time.perf_counter() - start

    # Evaluate after stopping the timer so RMSE does not affect the
    # one-epoch training measurement.
    model.eval()
    valid_true, valid_pred = [], []
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb, yb = xb.to(device), yb.to(device)
            valid_true.append(yb)
            valid_pred.append(model(xb).squeeze(-1))
    validation_rmse = metrics(
        torch.cat(valid_true), torch.cat(valid_pred)
    )[1]

    return {
        "batch_size": batch_size,
        "examples": len(train_loader.dataset),
        "training_rounds": len(train_loader),
        "one_epoch_seconds": elapsed,
        "validation_RMSE": validation_rmse,
    }

if QUICK_RUN:
    print("Run Part A on an NVIDIA GPU to populate the timing comparison.")
    batch_timing_summary = pd.DataFrame()
else:
    batch_timing_summary = pd.DataFrame([
        time_one_training_epoch(batch_size)
        for batch_size in BATCH_SIZES
    ])
    display(batch_timing_summary.round(3))

    fig, ax = plt.subplots(figsize=(7.5, 4))
    bars = ax.bar(
        batch_timing_summary.batch_size.astype(str),
        batch_timing_summary.one_epoch_seconds,
        color=["#94a3b8", "#76b900", "#5f9400"],
    )
    ax.bar_label(bars, fmt="%.2f s", padding=3)
    ax.set_title("One HQNN training epoch (128 examples)")
    ax.set_xlabel("Batch size")
    ax.set_ylabel("Training time (seconds)")
    ax.grid(axis="y", alpha=0.25)
    plt.show()

### Part B — hold the batch size fixed and vary the learning rate

This experiment fixes `batch_size=128`, so every epoch contains one training round and one optimizer update. Each learning-rate run starts from the same weights and uses the same data order. Training for 20 epochs gives each setting 20 updates, and the plot shows validation RMSE after each one.

The chosen rates deliberately span a wide range. They are examples for observation, not universal recommendations: useful values depend on the model, optimizer, data, and batch size.

> **How to read the plot:** A downward validation-RMSE curve indicates improvement. A nearly flat curve suggests that the learning rate may be too small for this update budget. A curve that drops quickly and then rises or oscillates suggests that the updates may be overshooting. Compare both the best RMSE reached and the final RMSE—an aggressive learning rate can look best briefly but finish worse.

In [ ]:
LEARNING_RATE_BATCH_SIZE = 128
LEARNING_RATE_TRAIN_SAMPLES = 128
LEARNING_RATE_EPOCHS = 20
LEARNING_RATES = (1e-4, 6e-3, 5e-2)

if QUICK_RUN:
    print("Run Part B on an NVIDIA GPU to populate the RMSE curves.")
    learning_rate_runs = []
    learning_rate_summary = pd.DataFrame()
else:
    learning_rate_runs = [
        run_single_qubit_experiment(
            name=f"learning rate = {learning_rate:g}",
            batch_size=LEARNING_RATE_BATCH_SIZE,
            learning_rate=learning_rate,
            epochs=LEARNING_RATE_EPOCHS,
            cudnn_enabled=True,
            train_samples=LEARNING_RATE_TRAIN_SAMPLES,
            evaluate_test=False,
        )
        for learning_rate in LEARNING_RATES
    ]

    learning_rate_summary = pd.DataFrame([
        {
            "learning_rate": run["learning_rate"],
            "updates": run["updates_per_epoch"] * run["epochs"],
            "best_validation_RMSE": run["history"].valid_rmse.min(),
            "final_validation_RMSE": run["history"].valid_rmse.iloc[-1],
            "seconds": run["elapsed_seconds"],
        }
        for run in learning_rate_runs
    ])
    display(learning_rate_summary.round(4))

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for run in learning_rate_runs:
        history = run["history"]
        ax.plot(
            history.epoch, history.valid_rmse, marker="o", markersize=4,
            label=f"learning rate = {run['learning_rate']:g}",
        )
    ax.set_title("Learning rate changes the validation-RMSE trajectory")
    ax.set_xlabel("Epoch (one optimizer update per epoch)")
    ax.set_ylabel("Validation RMSE")
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

configure_cudnn(True)

### Solution comments

<details>
<summary><strong>Show solution comments for the batching and learning-rate exercise</strong></summary>

**Part A — batch size:** All three runs process the same 128 examples, but they do not perform the same amount of optimization. Batch sizes 1, 64, and 128 produce 128, 2, and 1 parameter updates, respectively. The larger batches should take much less time because they replace many short hybrid training rounds with a few larger ones.

The validation RMSE values are worth inspecting because they expose the optimization tradeoff. In the reference run, batch sizes 1, 64, and 128 produced RMSEs of approximately 0.140, 0.425, and 0.430. However, they are not a fair measure of which batch size is most accurate: the `batch_size=1` model has received 128 updates, while the `batch_size=128` model has received only one. A more controlled accuracy study would compare equal update counts and tune the learning rate for each batch size, while also reporting that the larger-batch runs then process more examples and perform more total work.

**Part B — learning rate:** In the reference run, `1e-4` changed the validation RMSE only slowly, `6e-3` produced a sustained decrease, and `5e-2` decreased quickly before rising again. The last curve demonstrates why both the best and final RMSE matter: a large learning rate can reach a good region quickly and then overshoot it. Learning-rate choices should be made with validation data, not the test set.

Exact values and timings will vary by GPU and software environment, but the update counts are fixed by $\lceil N/B\rceil$.

</details>

---
## 5. A scalable SPSA quantum gradient

Mini-batching reduces the number of training rounds in an epoch, but every remaining round still computes a gradient. Our next goal is to make that gradient calculation as fast as the method and hardware allow—particularly the quantum portion, which requires additional circuit evaluations rather than ordinary PyTorch backpropagation.

Recall that the parameter-shift method used in Part 1 is exact, but a circuit with $P$ trainable angles requires $2P$ shifted logical circuit evaluations per example. That scaling becomes increasingly expensive as the quantum layer grows. Here we retain the original parameter-shift code as the baseline implementation, but focus this extension on **SPSA** as a lower-cost training gradient.

SPSA estimates every angle derivative from only two simultaneous perturbations. We will also submit those two perturbed circuits through CUDA-Q's MQPU backend so independent work can use multiple QPUs when the hardware provides them. On this one-QPU machine the MQPU path remains correct, but parallel acceleration is not expected.

### SPSA with MQPU dispatch

For a random perturbation vector $\Delta$ with entries in $\{-1,+1\}$ and a small perturbation magnitude $c$, SPSA estimates component $i$ of the gradient as

$$
\widehat{g}_i(\theta) =
\frac{f(\theta+c\Delta)-f(\theta-c\Delta)}{2c\Delta_i}.
$$

The two function values perturb **all** angles simultaneously, so the evaluation count does not grow with the number of angles. To make this concrete: a circuit with 18 trainable angles requires 36 shifted circuit evaluations per backward pass with parameter-shift — two per angle — while SPSA requires exactly 2, regardless of how many angles the circuit contains. One random direction is noisy, but independent $+1/-1$ directions make unwanted cross-terms cancel in expectation. Across training updates, a stochastic optimizer can accumulate these inexpensive descent estimates. The perturbation $c$ controls a bias–noise tradeoff: making it smaller reduces finite-difference bias for a smooth objective, but can amplify numerical or shot noise.

The positive and negative evaluations are also independent. In this context, each "QPU" is a GPU-simulated quantum processor instance managed by CUDA-Q; dispatching to multiple QPUs means routing independent circuit evaluations to separate simulator instances that can run concurrently, rather than sequentially on a single instance. The implementation below submits both through `cudaq.observe_async` on the MQPU target before collecting either result. With multiple CUDA-Q QPUs, those jobs can run concurrently. This machine exposes one QPU, so the code path is valid but is not expected to accelerate this small circuit.

SPSA was established by J. C. Spall in [*Multivariate Stochastic Approximation Using a Simultaneous Perturbation Gradient Approximation*](https://doi.org/10.1109/9.119632), IEEE Transactions on Automatic Control (1992).



**Exercise 3: Implement an MQPU-dispatched SPSA gradient**

The cell below implements SPSA gradient estimation with optional MQPU dispatch. Complete the `TODO` items to build the full implementation.

**a. Random direction.** Generate an independent Rademacher perturbation—$-1$ or $+1$ with equal probability—for every angle in every example.

**b. Simultaneous shifts.** Construct `theta_plus = theta + c * delta` and `theta_minus = theta - c * delta`.

**c. MQPU evaluation.** Submit both shifted evaluations before waiting for either result. Assign jobs to available QPU IDs in round-robin order and use both results in the SPSA formula.

**d. End-to-end evidence.** Train the original HQNN for several updates on one fixed mini-batch. Report the initial and final MSE, percentage change in MSE, and fully connected layer gradient norm.

</div>

### Reference solution — inject SPSA into the main HQNN

A standalone estimator does not affect training until it is called from the autograd bridge. The integration point is the original `QuantumFunction.backward`: `SPSAQuantumFunction.backward` below calls the MQPU-dispatched SPSA evaluator and then multiplies the estimated local derivative by PyTorch's upstream `grad_output`.

Only one model-construction line changes. In `Hybrid_QNN.__init__`, replace

```python
self.quantum = QuantumLayer(qubit_count, hamiltonian, shift)
```

with

```python
self.quantum = SPSAQuantumLayer(
    qubit_count, hamiltonian, perturbation=0.1, use_mqpu=True
)
```

The LSTM, fully connected layer, forward circuit, loss, optimizer, and training loop remain unchanged. The code keeps a synchronous fallback for CPU development, while `use_mqpu=True` submits both SPSA shifts asynchronously on NVIDIA hardware. The `TODO` comments correspond to Exercise 3(a–d); their completed solution remains executable here.

In [ ]:
# Exercise 3 — implement the SPSA gradient and optional MQPU dispatch.
# TODO a: generate an independent -1/+1 direction for every angle.
# TODO b: construct simultaneous positive and negative perturbations.
# TODO c: submit both shifted evaluations before collecting either result.
# TODO c: apply the SPSA formula while preserving (batch, angles).
# TODO d: replace the original quantum layer in Hybrid_QNN with SPSAQuantumLayer.
# TODO d: demonstrate end-to-end training on a fixed mini-batch.

raise NotImplementedError("Implement the Exercise 3 TODOs.")


---
## 6. Add qubits for a multi-step forecast

We now change the application from predicting one next value to predicting the next three values. Set `LOOKBACK=6` and `HORIZON=3`, while preserving the original train, validation, and test boundaries.

This is deliberately a small, runnable demonstration of how a quantum layer can return several outputs. It is not intended to be a highly optimized forecasting model or evidence that the quantum model outperforms a simple time-series baseline. The configuration uses three qubits, three data-dependent circuit angles, three readouts, a 16-unit LSTM, 800 sampled training windows, and ten epochs.

- the linear layer produces one angle per forecast step;
- qubit $i$ receives `h` followed by `ry(theta[i])`;
- a CNOT chain lets information influence neighboring forecast readouts; and
- $Z_i$ supplies the prediction for step $t+i+1$.

The three readouts do not require three additional parameters: the same three-angle circuit state is measured through $Z_0$, $Z_1$, and $Z_2$. The angles are generated by the classical network for each input sample; the LSTM and linear layer still contain the trainable model weights. Using one qubit and one angle per horizon step is a transparent simplification for this lesson, not a general requirement of multi-step forecasting.

### Batch the multi-qubit training

This quantum layer has three angles. Its exact backward pass therefore calls the batched quantum evaluator six times in every optimizer round—one positive and one negative shift for each angle. Every shifted circuit evaluation returns all three readouts. With 800 training examples, `batch_size=128` gives $\lceil 800/128 \rceil=7$ optimizer rounds per epoch.

For a short classroom run, use ten epochs and a mild cosine learning-rate decay from `2e-2` toward `1e-2`. The best-validation checkpoint is restored before testing, so the final epoch does not automatically determine the reported result.

On the development GPU, this 70-update configuration completed in about 55 seconds. Its purpose is to make the multi-output forward and backward paths concrete while remaining practical to execute. Accuracy will vary by run and should be interpreted alongside a simple persistence forecast, especially because this three-step temperature series is very smooth.

In [ ]:
LOOKBACK = 6
HORIZON = 3
MULTI_BATCH_SIZE = 128
MULTI_EVAL_BATCH_SIZE = 256

X_train_h, y_train_h = make_windows(train_series, LOOKBACK, HORIZON)
X_valid_h, y_valid_h = make_windows(valid_series, LOOKBACK, HORIZON)
X_test_h, y_test_h = make_windows(test_series, LOOKBACK, HORIZON)

rng = np.random.default_rng(SEED)
train_h_indices = np.sort(
    rng.choice(len(y_train_h), MAX_TRAIN_SAMPLES, replace=False)
)
valid_h_indices = np.linspace(
    0, len(y_valid_h) - 1, MAX_VALID_SAMPLES, dtype=int
)

X_train_h, y_train_h = X_train_h[train_h_indices], y_train_h[train_h_indices]
X_valid_h, y_valid_h = X_valid_h[valid_h_indices], y_valid_h[valid_h_indices]
train_loader_h = to_loader(
    X_train_h, y_train_h, batch_size=MULTI_BATCH_SIZE, shuffle=True
)
valid_loader_h = to_loader(
    X_valid_h, y_valid_h, batch_size=MULTI_BATCH_SIZE, shuffle=False
)
test_loader_h = to_loader(
    X_test_h, y_test_h, batch_size=MULTI_EVAL_BATCH_SIZE, shuffle=False
)

print(
    f"Multi-step shapes: X_train={X_train_h.shape}, y_train={y_train_h.shape}"
)
print(f"Full multi-step test windows: {len(y_test_h)}")
print(
    f"Batch size {MULTI_BATCH_SIZE}: {len(train_loader_h)} training rounds "
    "per epoch"
)

### Several readouts and a vector-Jacobian product

A single Hamiltonian is constructed from all three $Z_i$ terms. After one broadcast `observe` call, each term expectation is extracted from the result. This avoids writing a separate Python observation loop for every output and gives CUDA-Q the complete observable at once. Backend execution details can still depend on the simulator, target, and shot configuration.

Entanglement means shifting one angle can change every output. The backward pass therefore forms a vector of output derivatives and contracts it with PyTorch's upstream gradient:

$$\frac{\partial L}{\partial \theta_i}=\sum_h\frac{\partial L}{\partial y_h}\frac{\partial y_h}{\partial \theta_i}.$$

This vector-Jacobian product is the multi-output version of the scalar multiplication used in Part 1.

In [ ]:
def build_multiqubit_kernel(qubit_count):
    @cudaq.kernel
    def kernel(qubit_count: int, thetas: np.ndarray):
        qubits = cudaq.qvector(qubit_count)
        for i in range(qubit_count):
            h(qubits[i])
            ry(thetas[i], qubits[i])
        for i in range(qubit_count - 1):
            x.ctrl(qubits[i], qubits[i + 1])
    return kernel


class MultiOutputQuantumFunction(Function):
    def __init__(self, qubit_count):
        self.kernel = build_multiqubit_kernel(qubit_count)
        self.qubit_count = qubit_count
        self.observables = [spin.z(i) for i in range(qubit_count)]
        self.hamiltonian = self.observables[0]
        for observable in self.observables[1:]:
            self.hamiltonian += observable

    def run(self, theta_vals):
        theta_host = theta_vals.detach().cpu().numpy()
        qubit_counts = [self.qubit_count] * len(theta_host)
        results = cudaq.observe(
            self.kernel, self.hamiltonian, qubit_counts, theta_host
        )
        outputs = [
            [result.expectation(observable) for observable in self.observables]
            for result in results
        ]
        return torch.tensor(outputs, device=theta_vals.device, dtype=theta_vals.dtype)

    @staticmethod
    def forward(ctx, thetas, quantum_circuit, shift):
        ctx.quantum_circuit = quantum_circuit
        ctx.shift = shift
        ctx.save_for_backward(thetas)
        return quantum_circuit.run(thetas)

    @staticmethod
    def backward(ctx, grad_output):
        (thetas,) = ctx.saved_tensors
        gradients = torch.zeros_like(thetas)

        for i in range(thetas.shape[1]):
            thetas_plus = thetas.clone()
            thetas_plus[:, i] += ctx.shift
            outputs_plus = ctx.quantum_circuit.run(thetas_plus)

            thetas_minus = thetas.clone()
            thetas_minus[:, i] -= ctx.shift
            outputs_minus = ctx.quantum_circuit.run(thetas_minus)

            output_derivative = (outputs_plus - outputs_minus) / 2.0
            gradients[:, i] = (grad_output * output_derivative).sum(dim=1)

        return gradients, None, None


class MultiOutputQuantumLayer(nn.Module):
    def __init__(self, qubit_count):
        super().__init__()
        self.quantum_circuit = MultiOutputQuantumFunction(qubit_count)
        self.register_buffer("shift", torch.tensor(torch.pi / 2))

    def forward(self, inputs):
        return MultiOutputQuantumFunction.apply(
            inputs, self.quantum_circuit, self.shift
        )


class MultiHorizonHQNN(nn.Module):
    """Part 1's LSTM pattern extended to several quantum outputs."""

    def __init__(self, horizon, input_size=1, hidden_size=16,
                 num_layers=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, horizon)
        self.quantum = MultiOutputQuantumLayer(horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        ann = self.fc(last)
        return self.quantum(ann)

In [ ]:
def evaluate_multiqubit_loader(model, loader):
    model.eval()
    truths, predictions = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            truths.append(yb)
            predictions.append(model(xb))
    return metrics(torch.cat(truths), torch.cat(predictions))


def train_multiqubit_with_decay(model, train_loader, valid_loader, epochs,
                                initial_lr=2e-2, final_lr=1e-2):
    """Train with mild cosine decay and restore the best validation state."""
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=initial_lr)
    # Learning-rate decay is added here. Step it once after each epoch.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=final_lr
    )
    best_state = copy.deepcopy(model.state_dict())
    best_valid_rmse = float("inf")
    best_epoch = 0
    rows = []

    for epoch in range(1, epochs + 1):
        print(f"Starting multi-qubit epoch {epoch}/{epochs} ({len(train_loader)} batches)", flush=True)
        learning_rate = optimizer.param_groups[0]["lr"]
        model.train()
        squared_error_sum = 0.0
        element_count = 0

        for batch_index, (xb, yb) in enumerate(train_loader, start=1):
            if batch_index == 1 or batch_index % 10 == 0 or batch_index == len(train_loader):
                print(f"  multi-qubit batch {batch_index}/{len(train_loader)} starting", flush=True)
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            prediction = model(xb)
            loss = criterion(prediction, yb)
            loss.backward()
            optimizer.step()
            squared_error_sum += ((prediction.detach() - yb) ** 2).sum().item()
            element_count += yb.numel()

        train_rmse = np.sqrt(squared_error_sum / element_count)
        valid_mse, valid_rmse, valid_r2 = evaluate_multiqubit_loader(
            model, valid_loader
        )
        rows.append({
            "epoch": epoch,
            "learning_rate": learning_rate,
            "train_rmse": train_rmse,
            "valid_mse": valid_mse,
            "valid_rmse": valid_rmse,
            "valid_r2": valid_r2,
        })

        if valid_rmse < best_valid_rmse:
            best_valid_rmse = valid_rmse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch:02d} | lr={learning_rate:.5f} | "
            f"train RMSE={train_rmse:.4f} | valid RMSE={valid_rmse:.4f}"
        )
        scheduler.step()

    model.load_state_dict(best_state)
    return model, pd.DataFrame(rows), best_epoch, best_valid_rmse


configure_cudnn(True)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
multi_model = MultiHorizonHQNN(HORIZON, hidden_size=16).to(device)

MULTI_EPOCHS = 10
MULTI_INITIAL_LR = 2e-2
MULTI_FINAL_LR = 1e-2

if QUICK_RUN:
    print("Skipping multi-qubit training: run this section with the NVIDIA GPU target.")
    multi_history = pd.DataFrame()
    multi_true = multi_pred = None
else:
    print(
        f"Training {MULTI_EPOCHS} epochs × {len(train_loader_h)} rounds "
        f"= {MULTI_EPOCHS * len(train_loader_h)} optimizer updates"
    )
    synchronize()
    multi_start = time.perf_counter()
    multi_model, multi_history, best_epoch, best_valid_rmse = (
        train_multiqubit_with_decay(
            multi_model, train_loader_h, valid_loader_h,
            epochs=MULTI_EPOCHS, initial_lr=MULTI_INITIAL_LR,
            final_lr=MULTI_FINAL_LR,
        )
    )
    synchronize()
    multi_elapsed = time.perf_counter() - multi_start
    multi_mse, multi_rmse, multi_r2, multi_true, multi_pred = test_model(
        multi_model, test_loader_h
    )
    display(multi_history[[
        "epoch", "learning_rate", "train_rmse", "valid_rmse"
    ]].round(4))

    fig, ax = plt.subplots(figsize=(7.8, 4.4))
    ax.plot(multi_history.epoch, multi_history.train_rmse,
            marker="o", label="Training RMSE")
    ax.plot(multi_history.epoch, multi_history.valid_rmse,
            marker="o", label="Validation RMSE")
    ax.set_title("Batched three-qubit HQNN with learning-rate decay")
    ax.set_xlabel(
        f"Epoch ({len(train_loader_h)} optimizer updates each)"
    )
    ax.set_ylabel("RMSE")
    ax.grid(alpha=0.25)

    lr_axis = ax.twinx()
    lr_axis.plot(multi_history.epoch, multi_history.learning_rate,
                 color="#76B900", linestyle="--", label="Learning rate")
    lr_axis.set_ylabel("Learning rate", color="#4d7f00")
    lines = ax.get_lines() + lr_axis.get_lines()
    ax.legend(lines, [line.get_label() for line in lines], loc="best")
    plt.show()

    print({
        "elapsed_seconds": multi_elapsed,
        "best_epoch": best_epoch,
        "best_validation_rmse": best_valid_rmse,
        "test_mse": multi_mse,
        "test_rmse": multi_rmse,
        "test_r2": multi_r2,
    })

In [ ]:
def per_horizon_metrics(y_true, y_pred):
    rows = []
    for horizon_index in range(y_true.shape[1]):
        mse, rmse, r2 = metrics(
            y_true[:, horizon_index], y_pred[:, horizon_index]
        )
        rows.append({
            "step_ahead": f"t+{horizon_index + 1}",
            "mse": mse, "rmse": rmse, "r2": r2,
        })
    return pd.DataFrame(rows)

def inverse_scale(values, data_min=15.473, data_max=28.258):
    return values * (data_max - data_min) + data_min

if multi_true is None:
    print("No multi-horizon training results to plot in CPU-only mode.")
else:
    horizon_results = per_horizon_metrics(multi_true, multi_pred)
    display(horizon_results)

    ground_truth = inverse_scale(multi_true.detach().cpu().numpy())
    predictions = inverse_scale(multi_pred.detach().cpu().numpy())
    time_index = np.arange(len(ground_truth))

    fig, axes = plt.subplots(HORIZON, 1, figsize=(11, 3 * HORIZON), sharex=True)
    for h in range(HORIZON):
        axes[h].plot(time_index, ground_truth[:, h], linewidth=1.0, label="Observed")
        axes[h].plot(
            time_index, predictions[:, h], linewidth=1.0,
            label=f"Predicted t+{h + 1}"
        )
        axes[h].set_ylabel("Temperature [°C]")
        axes[h].grid(alpha=0.3)
        axes[h].legend()
    axes[0].set_title("Three-output HQNN over the full test period")
    axes[-1].set_xlabel("Test time step")
    plt.tight_layout()
    plt.show()

### The exercise below asks you to use an AI coding agent to design and implement a scaled HQNN

Before opening any AI tool, write your answers to the following questions in a new markdown cell — this gives you a specification to evaluate the result against:

1. **Dataset and target:** What dataset will you use, and what is the prediction target?
2. **Success metric:** What RMSE (root mean squared error — the square root of the average squared difference between predicted and actual values) would you consider meaningfully better than the Part 1 baseline? Write the number down.
3. **Failure signature:** What would a barren plateau look like in the training curve for this architecture? What would overfitting look like?
4. **Scale constraint:** What is the largest number of qubits you will allow the agent to propose, and why?

Having a written specification before you generate makes it possible to evaluate what you got — and to identify what to change in your prompt if the result doesn't meet it.



**<Open-ended exercise: Autoresearch a scaled CUDA-Q HQNN**

**Problem.** Find a public supervised-learning dataset with a clear prediction target. Ask an AI coding agent to inspect the data and plan a larger CUDA-Q HQNN. Fix the train, validation, and test splits before experimentation.

**Build.** Have the agent implement and verify a model that is meaningfully larger than the lesson examples. Consider mini-batching, cuDNN, learning-rate schedules, checkpointing, profiling, SPSA, and parallel quantum-gradient evaluation with MQPU when the hardware supports it. Require shape checks, a successful forward and backward pass, finite gradients, and a measured baseline.

**Autoresearch.** Adapt [Karpathy's Autoresearch project](https://github.com/karpathy/autoresearch): set a fixed wall-clock or trial budget, keep the splits and metric fixed, let the agent test training changes, retain changes only when validation performance improves, and log every result. Do not use the test set to select experiments.

**Report.** Submit the dataset and target, original plan, baseline and best configurations, experiment log, runtime, validation and test metrics, and a short analysis of which optimizations helped. Distinguish measured speedups from expectations, state the available GPU and QPU resources, and identify at least one limitation or follow-up experiment.

</div>

### After the agent: evaluate against your specification

Before reporting results, work through these checks and write a short note on each:

- **RMSE threshold:** Does the final validation RMSE meet the number you wrote above? If not, is the gap meaningful or within noise?
- **Training curve:** Compare the training and validation curves to what you predicted. What does the curve tell you about whether this architecture is trainable? If the behavior surprises you — converges faster, slower, or differently than expected — explain why.
- **Gradient sanity check:** If the agent used a gradient-based optimizer, verify the gradient estimates are directionally correct: evaluate the loss at a small positive and negative perturbation of one parameter and confirm the sign of the estimated gradient is consistent with the observed loss change.
- **Architecture justification:** Does the qubit count and circuit depth the agent proposed match the scale of the problem? Write one sentence explaining why that choice is or isn't appropriate for the dataset you selected.

Use these notes to identify what you would change in your specification or prompt for the next iteration.

## Conclusion

After completing this lesson, you now know a number of ways you can improve or optimize a HQNN workflow to improve performance or introduce meaningful tradeoffs.  If you completed the agentic AI exercise, you also learned how to combine these different techniques and how much nuance is required to land on an optimal workflow.

| Step | What stayed fixed | What changed | Evidence to inspect |
|---|---|---|---|
| cuDNN | temperature series, input precision, seeded weights, CUDA GPU | classical execution backend and LSTM workload | baseline and scaled one-pass timings |
| Batching and learning rate | one-qubit HQNN, data subset, seeded weights | examples per round and optimizer step size | rounds per epoch, runtime, validation-RMSE trajectories |
| Scalable quantum gradient | parameterized circuit and PyTorch bridge | SPSA estimator with MQPU-dispatched shifts | fixed-batch MSE change, gradient norm, available QPUs, and evaluation count |
| Multi-step extension | LSTM pattern and PyTorch bridge | three `h`–`ry` qubits, three readouts, entanglement, batch size, and learning rate | rounds per epoch, RMSE trajectory, per-horizon accuracy, and vector gradients |

Recall that HQNNs are flexible tools that can be used for multiple QML applications.  Many of these principles are agnostic of the application and can be helpful for developing HQNNs for unsupervised learning, reinforcement learning, and other more advanced QML applications.